In [1]:
import pandas as pd
import os

# Carica i file
combined_file = "all_africa_combined_20260429_104943.csv"
summary_file = "summary_20260429_104943.csv"

if not os.path.exists(combined_file) or not os.path.exists(summary_file):
    print("File mancanti. Assicurati che i CSV siano nella cartella corrente.")
    exit()

df = pd.read_csv(combined_file, low_memory=False)
summary = pd.read_csv(summary_file)

# ==================== Statistiche dal COMBINED (punti unici) ====================
# Deduplicazione per place_id a livello globale (per gestire eventuali sovrapposizioni tra città)
# Il CSV combinato dovrebbe già essere deduplicato per città, ma controlliamo.
print("=== DAL FILE COMBINATO (punti unici) ===")
print(f"Righe totali pre-dedup: {len(df)}")
unique_df = df.drop_duplicates(subset='place_id')
print(f"Righe dopo deduplica su place_id: {len(unique_df)}")

# Fonte
osm = unique_df[unique_df['source'] == 'osm']
google = unique_df[unique_df['source'] == 'google']
print(f"\nOSM: {len(osm)}")
print(f"Google: {len(google)}")

# Categoria
filling = unique_df[unique_df['category'] == 'filling']
reseller = unique_df[unique_df['category'] == 'reseller']
print(f"\nFilling: {len(filling)}")
print(f"Reseller: {len(reseller)}")

# Lingue di Google (solo google)
if len(google) > 0:
    print("\nLingue Google (punti unici):")
    lang_counts = google['language'].value_counts()
    for lang, cnt in lang_counts.items():
        lang_label = lang if lang != "" else "(vuoto/i non mappato)"
        print(f"  {lang_label}: {cnt} ({cnt/len(google)*100:.1f}%)")
    # Controlla la somma delle lingue rispetto al totale Google
    total_google = len(google)
    # somma dei conteggi potrebbe essere diversa se ci sono doppioni? no, è un conteggio per lingua
    # Verifica la somma: dovrebbe essere uguale a len(google) perché ogni punto ha una lingua
    # se ci sono NaN, vengono ignorati dalla value_counts ma il totale resta uguale? value_counts esclude NaN.
    # Controlliamo se ci sono NaN
    nan_count = google['language'].isna().sum()
    empty_count = (google['language'] == '').sum()
    if nan_count > 0:
        print(f"  Valori NaN: {nan_count}")
    if empty_count > 0:
        print(f"  Stringhe vuote: {empty_count}")

# ==================== Statistiche dal SUMMARY ====================
print("\n=== DAL FILE SUMMARY ===")
print(f"Numero di città con risultati: {len(summary)}")

# Somma delle colonne
total_points_sum = summary['Total_Points'].sum()
total_osm_sum = summary['Total_OSM'].sum()
total_google_sum = summary['Total_Google'].sum()
total_filling_sum = summary['Total_Filling'].sum()
total_reseller_sum = summary['Total_Reseller'].sum()

print(f"Somma Total_Points: {total_points_sum}")
print(f"Somma Total_OSM: {total_osm_sum}")
print(f"Somma Total_Google: {total_google_sum}")
print(f"Somma Total_Filling: {total_filling_sum}")
print(f"Somma Total_Reseller: {total_reseller_sum}")

# Città uniche nel summary
print(f"\nCittà uniche nel summary: {summary['City'].nunique()}")

# Paesi (controlliamo quanti)
print(f"Paesi unici nel summary: {summary['Country'].nunique()}")

# I 5 paesi con più punti (aggregazione)
country_totals = summary.groupby('Country')['Total_Points'].sum().sort_values(ascending=False)
print("\nTotale punti per Paese (dal summary, ordinato):")
print(country_totals.to_string())

# Top 5 quota
top5 = country_totals.head(5).sum()
print(f"\nTop 5 paesi: {top5} punti, {top5/total_points_sum*100:.1f}% del totale")

# Paesi con meno di 10 punti (sul summary)
low_countries = country_totals[country_totals < 10]
print(f"\nPaesi con <10 punti: {len(low_countries)}")
print(low_countries.to_string())

# Verifica incrociata: i punti del summary possono essere superiori a quelli deduplicati globalmente a causa di sovrapposizioni
print("\n=== NOTE ===")
print(f"Punti unici (deduplicati) nel combinato: {len(unique_df)}")
print(f"Somma punti dal summary: {total_points_sum}")
print("Se il summary ha più punti, significa che ci sono duplicati tra città (es. Lesotho, Sudan).")

=== DAL FILE COMBINATO (punti unici) ===
Righe totali pre-dedup: 3708
Righe dopo deduplica su place_id: 3342

OSM: 550
Google: 2792

Filling: 1009
Reseller: 2333

Lingue Google (punti unici):
  en: 2630 (94.2%)
  fr: 111 (4.0%)
  pt: 23 (0.8%)
  sw: 22 (0.8%)
  ar: 6 (0.2%)

=== DAL FILE SUMMARY ===
Numero di città con risultati: 88
Somma Total_Points: 3708
Somma Total_OSM: 658
Somma Total_Google: 3050
Somma Total_Filling: 1187
Somma Total_Reseller: 2521

Città uniche nel summary: 88
Paesi unici nel summary: 44

Totale punti per Paese (dal summary, ordinato):
Country
SOUTH_AFRICA                774
KENYA                       471
NIGERIA                     403
GHANA                       287
UGANDA                      214
COTE_DIVOIRE                189
ZIMBABWE                    169
ZAMBIA                      159
TANZANIA                    152
CAMEROON                    120
MALAWI                       95
RWANDA                       86
BENIN                        69
TOGO      